# 11 Wetter und Chancenqualitaet analysieren

Dieses Notebook implementiert BD-23, BD-24 und BD-25. Es untersucht, ob Wetterbedingungen eher mit Chancenqualitaet (`xg_per_shot`) als mit reiner Schussanzahl oder Toren zusammenhaengen, und bereitet anschliessend ein sauberes Regressions-Dataset fuer den Vergleich Wetter vs. Teamstaerke vor.

Input:
- `data/gold/team_match_analysis_dataset.csv`

Outputs:
- `outputs/figures/temperature_vs_xg_per_shot.png`
- `outputs/figures/feels_like_vs_xg_per_shot.png`
- `outputs/figures/xg_by_temperature_group.png`
- `outputs/figures/xg_rain_vs_no_rain.png`
- `outputs/figures/shots_rain_vs_no_rain.png`
- `outputs/tables/bd23_chance_quality_summary.csv`
- `outputs/tables/bd24_model_dataset.csv`
- `outputs/tables/bd24_model_variables.csv`
- `outputs/tables/bd24_model_missing_values.csv`


## Methodischer Ansatz

Die Analyse arbeitet auf Team-Match-Ebene, weil xG, Shots und Tore aus Sicht eines Teams entstehen. Chancenqualitaet wird bewusst getrennt vom Volumen betrachtet: `xg_per_shot` misst, wie gut die durchschnittliche Chance war, waehrend `shots` die Menge der Abschluesse beschreibt.

Fuer Regen werden Niederschlaege bis 0.5 mm als `kein/kaum Regen` zusammengefasst. Damit dominieren minimale Messwerte wie 0.1 mm nicht die Regen-Gruppe. Fuer BD-24 wird kein finales Vorhersagemodell trainiert; das Notebook vergleicht erklaerend, ob Wetter oder Teamstaerke mehr Varianz in `team_xg` beschreiben.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project_paths import project_root

ROOT = project_root()
DATA_PATH = ROOT / 'data' / 'gold' / 'team_match_analysis_dataset.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RAIN_THRESHOLD_MM = 0.5
RAIN_GROUP_ORDER = ['kein/kaum Regen', 'messbarer Regen']

TEMPERATURE_VS_XG_PER_SHOT_PATH = FIGURE_DIR / 'temperature_vs_xg_per_shot.png'
FEELS_LIKE_VS_XG_PER_SHOT_PATH = FIGURE_DIR / 'feels_like_vs_xg_per_shot.png'
XG_BY_TEMPERATURE_GROUP_PATH = FIGURE_DIR / 'xg_by_temperature_group.png'
XG_RAIN_VS_NO_RAIN_PATH = FIGURE_DIR / 'xg_rain_vs_no_rain.png'
SHOTS_RAIN_VS_NO_RAIN_PATH = FIGURE_DIR / 'shots_rain_vs_no_rain.png'
SUMMARY_PATH = TABLE_DIR / 'bd23_chance_quality_summary.csv'
MODEL_DATASET_PATH = TABLE_DIR / 'bd24_model_dataset.csv'
MODEL_VARIABLES_PATH = TABLE_DIR / 'bd24_model_variables.csv'
MODEL_MISSING_VALUES_PATH = TABLE_DIR / 'bd24_model_missing_values.csv'
MODEL_COMPARISON_PATH = TABLE_DIR / 'model_comparison.csv'
MODEL_INCREMENTAL_COMPARISON_PATH = TABLE_DIR / 'model_incremental_comparison.csv'
MODEL_COEFFICIENTS_TABLE_PATH = TABLE_DIR / 'model_coefficients.csv'
MODEL_R2_COMPARISON_PATH = FIGURE_DIR / 'model_r2_comparison.png'
MODEL_COEFFICIENTS_PATH = FIGURE_DIR / 'model_coefficients.png'

required_columns = [
    'match_id',
    'team_id',
    'short_name',
    'team_name',
    'xg',
    'shots',
    'xg_per_shot',
    'team_score',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'temperature_group',
    'weather_missing_flag',
]

plot_colors = {
    'quality': '#3569a8',
    'xg': '#2f7d58',
    'rain': '#456990',
    'shots': '#8a5a30',
}


## Daten laden und validieren

Das Notebook liest nur das Gold-Dataset. Die Checks stellen sicher, dass die Analyse auf einer eindeutigen Team-Match-Struktur und vollstaendigen Wetterdaten laeuft.


In [ ]:
assert DATA_PATH.exists(), f'Missing required input file: {DATA_PATH}'

team_match_df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in required_columns if column not in team_match_df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

analysis_df = team_match_df[required_columns].copy()
analysis_df['rain_group'] = np.where(
    analysis_df['rain_mm'] > RAIN_THRESHOLD_MM,
    RAIN_GROUP_ORDER[1],
    RAIN_GROUP_ORDER[0],
)
analysis_df['rain_group'] = pd.Categorical(
    analysis_df['rain_group'],
    categories=RAIN_GROUP_ORDER,
    ordered=True,
)
analysis_df['xg_per_shot_check'] = np.where(
    analysis_df['shots'] > 0,
    analysis_df['xg'] / analysis_df['shots'],
    0.0,
)

required_metrics = [
    'xg',
    'shots',
    'xg_per_shot',
    'team_score',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'temperature_group',
]
missing_metric_values = analysis_df[required_metrics].isna().sum()
assert missing_metric_values.sum() == 0, f'Unexpected missing values: {missing_metric_values.to_dict()}'
assert not analysis_df.duplicated(['match_id', 'team_id']).any(), 'Expected one row per match/team pair.'
assert not analysis_df['weather_missing_flag'].any(), 'Weather data still contains missing weather rows.'
assert analysis_df['temperature_group'].nunique() >= 2, 'Expected at least two temperature groups for comparison.'
assert analysis_df['rain_group'].nunique() == 2, 'Expected both rain groups for comparison.'

max_xg_per_shot_delta = (analysis_df['xg_per_shot'] - analysis_df['xg_per_shot_check']).abs().max()
assert max_xg_per_shot_delta < 1e-9, f'xg_per_shot does not match xg / shots. Max delta: {max_xg_per_shot_delta}'

analysis_df.head()


## Nullwerte in xG und Shots

Vor den Grafiken wird geprueft, ob echte Nullwerte in den Chancenmetriken vorkommen. Das ist wichtig, weil ein einzelnes Team-Spiel mit `xg=0.0` und `shots=0` im Scatterplot als Punkt bei 0 erscheinen kann.


In [ ]:
zero_xg_or_shots_rows = analysis_df.loc[
    (analysis_df['xg'] == 0) | (analysis_df['shots'] == 0) | (analysis_df['xg_per_shot'] == 0),
    [
        'match_id',
        'team_name',
        'xg',
        'shots',
        'xg_per_shot',
        'team_score',
        'temperature_c',
        'temperature_group',
        'rain_group',
    ],
]

zero_metric_summary = {
    'zero_xg_team_matches': int((analysis_df['xg'] == 0).sum()),
    'zero_shot_team_matches': int((analysis_df['shots'] == 0).sum()),
    'zero_xg_per_shot_team_matches': int((analysis_df['xg_per_shot'] == 0).sum()),
    'zero_temperature_team_matches': int((analysis_df['temperature_c'] == 0).sum()),
    'zero_feels_like_team_matches': int((analysis_df['feels_like_c'] == 0).sum()),
}

assert zero_metric_summary['zero_temperature_team_matches'] == 0, 'Unexpected 0-degree temperature rows.'
zero_xg_or_shots_rows


## Kennzahlen zu Chancenqualitaet, Volumen und Toren

Die Summary trennt drei Ebenen: Chancenqualitaet (`xg_per_shot`), Chance-Volumen (`shots`) und Ergebnis (`team_score`). Dadurch wird BD-23 nicht auf reine Toranzahl reduziert.


In [ ]:
def correlation_row(dataframe, weather_column, metric_column, metric_label):
    values = dataframe[[weather_column, metric_column]].dropna()
    return {
        'weather_variable': weather_column,
        'metric': metric_label,
        'n_team_matches': int(len(values)),
        'correlation': round(float(values[weather_column].corr(values[metric_column])), 3),
        'metric_mean': round(float(values[metric_column].mean()), 3),
        'metric_median': round(float(values[metric_column].median()), 3),
    }

summary_rows = []
for weather_column in ['temperature_c', 'feels_like_c', 'rain_mm']:
    summary_rows.extend([
        correlation_row(analysis_df, weather_column, 'xg_per_shot', 'chance_quality_xg_per_shot'),
        correlation_row(analysis_df, weather_column, 'shots', 'shot_volume'),
        correlation_row(analysis_df, weather_column, 'xg', 'team_xg'),
        correlation_row(analysis_df, weather_column, 'team_score', 'goals'),
    ])

rain_summary = (
    analysis_df.sort_values('rain_group')
    .groupby('rain_group', observed=False)
    .agg(
        team_matches=('match_id', 'size'),
        xg_per_shot_mean=('xg_per_shot', 'mean'),
        xg_per_shot_median=('xg_per_shot', 'median'),
        xg_mean=('xg', 'mean'),
        xg_median=('xg', 'median'),
        shots_mean=('shots', 'mean'),
        shots_median=('shots', 'median'),
        goals_mean=('team_score', 'mean'),
        goals_median=('team_score', 'median'),
    )
    .reset_index()
)

correlation_summary = pd.DataFrame(summary_rows)
correlation_summary.to_csv(SUMMARY_PATH, index=False)
correlation_summary


## Plot-Helfer

Die Helfer halten Achsen, Trendlinie und Output-Speicherung einheitlich. Scatterplots zeigen kontinuierliche Wetterwerte. Der Temperaturgruppenvergleich bleibt als Boxplot, weil er fuer gruppierte Temperatur-Buckets kompakter und praesentierbarer ist.


In [ ]:
def save_scatter_with_trend(dataframe, x_column, y_column, title, xlabel, ylabel, output_path, color):
    plot_df = dataframe[[x_column, y_column]].dropna()
    correlation = plot_df[x_column].corr(plot_df[y_column])

    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=160)
    ax.scatter(
        plot_df[x_column],
        plot_df[y_column],
        alpha=0.6,
        s=36,
        color=color,
        edgecolor='white',
        linewidth=0.4,
    )

    if plot_df[x_column].nunique() > 1:
        slope, intercept = np.polyfit(plot_df[x_column], plot_df[y_column], 1)
        x_values = np.linspace(plot_df[x_column].min(), plot_df[x_column].max(), 100)
        ax.plot(x_values, slope * x_values + intercept, color='#222222', linewidth=2)

    ax.set_title(title, loc='left', fontweight='bold', pad=24)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.text(
        0.02,
        0.96,
        f'n={len(plot_df)} | r={correlation:.2f}',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=10,
        bbox={'facecolor': 'white', 'edgecolor': '#dddddd', 'boxstyle': 'round,pad=0.35'},
    )
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()

    return {'n': int(len(plot_df)), 'correlation': round(float(correlation), 3)}


def save_boxplot(dataframe, group_column, value_column, order, title, xlabel, ylabel, output_path, color):
    available_order = [group for group in order if group in set(dataframe[group_column].dropna())]
    values = [dataframe.loc[dataframe[group_column] == group, value_column].dropna() for group in available_order]
    counts = [len(series) for series in values]

    assert available_order, f'No groups available for {group_column}'
    assert all(count > 0 for count in counts), f'Empty group found for {group_column}'

    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=160)
    box = ax.boxplot(values, labels=available_order, patch_artist=True, showmeans=True)

    for patch in box['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
    for median in box['medians']:
        median.set_color('#222222')
        median.set_linewidth(2)
    for mean in box['means']:
        mean.set_marker('o')
        mean.set_markerfacecolor('white')
        mean.set_markeredgecolor('#222222')

    y_top = max(series.max() for series in values)
    y_bottom = min(series.min() for series in values)
    label_offset = max((y_top - y_bottom) * 0.04, y_top * 0.02, 0.02)
    for index, count in enumerate(counts, start=1):
        ax.text(index, y_top + label_offset, f'n={count}', ha='center', va='bottom', fontsize=9)

    ax.set_title(title, loc='left', fontweight='bold', pad=24)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(bottom=max(0, y_bottom - label_offset), top=y_top + label_offset * 4)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()

    return pd.DataFrame({
        group_column: available_order,
        'count': counts,
        'mean': [round(float(series.mean()), 3) for series in values],
        'median': [round(float(series.median()), 3) for series in values],
    })


## Temperatur vs. xG pro Schuss

Dieser Scatterplot ist die Kernfrage fuer Chancenqualitaet: Er zeigt, ob waermere Bedingungen mit besseren oder schlechteren durchschnittlichen Abschlusspositionen zusammenfallen.


In [ ]:
temperature_quality_relation = save_scatter_with_trend(
    analysis_df,
    x_column='temperature_c',
    y_column='xg_per_shot',
    title='Temperatur vs. xG pro Schuss',
    xlabel='Temperatur bei Anpfiff in Grad Celsius',
    ylabel='xG pro Schuss',
    output_path=TEMPERATURE_VS_XG_PER_SHOT_PATH,
    color=plot_colors['quality'],
)
temperature_quality_relation


## Gefuehlte Temperatur vs. xG pro Schuss

Die gefuehlte Temperatur beruecksichtigt weitere Wettereffekte. Sie kann deshalb fuer koerperliche Belastung informativer sein als die reine Lufttemperatur.


In [ ]:
feels_like_quality_relation = save_scatter_with_trend(
    analysis_df,
    x_column='feels_like_c',
    y_column='xg_per_shot',
    title='Gefuehlte Temperatur vs. xG pro Schuss',
    xlabel='Gefuehlte Temperatur bei Anpfiff in Grad Celsius',
    ylabel='xG pro Schuss',
    output_path=FEELS_LIKE_VS_XG_PER_SHOT_PATH,
    color=plot_colors['quality'],
)
feels_like_quality_relation


## xG nach Temperaturgruppe

Der Boxplot zeigt Team-xG nach Temperaturgruppe. Das ist keine reine Chancenqualitaet, sondern verbindet Qualitaet und Anzahl der Abschluesse. Er wird deshalb zusammen mit `xg_per_shot` und `shots` gelesen.


In [ ]:
temperature_order = ['cold', 'mild', 'warm', 'hot']
observed_temperature_order = [group for group in temperature_order if group in set(analysis_df['temperature_group'])]

xg_by_temperature_group = save_boxplot(
    analysis_df,
    group_column='temperature_group',
    value_column='xg',
    order=observed_temperature_order,
    title='xG nach Temperaturgruppe',
    xlabel='Temperaturgruppe',
    ylabel='xG pro Team-Spiel',
    output_path=XG_BY_TEMPERATURE_GROUP_PATH,
    color=plot_colors['xg'],
)
xg_by_temperature_group


## xG nach Regen-Gruppe

Der Boxplot vergleicht `kein/kaum Regen` (bis 0.5 mm) und `messbarer Regen` (mehr als 0.5 mm). Das ist praesentationsfreundlicher als ein Scatterplot, weil sehr viele Team-Spiele bei 0 mm liegen.


In [ ]:
xg_by_rain_group = save_boxplot(
    analysis_df,
    group_column='rain_group',
    value_column='xg',
    order=RAIN_GROUP_ORDER,
    title='xG nach Regen-Gruppe',
    xlabel='Regenbedingung',
    ylabel='xG pro Team-Spiel',
    output_path=XG_RAIN_VS_NO_RAIN_PATH,
    color=plot_colors['rain'],
)
xg_by_rain_group


## Shots nach Regen-Gruppe

Die Schusszahl zeigt, ob messbarer Regen eher mit dem Abschlussvolumen zusammenhaengt. Wegen der kleinen Regen-Gruppe bleibt der Vergleich explorativ.


In [ ]:
shots_by_rain_group = save_boxplot(
    analysis_df,
    group_column='rain_group',
    value_column='shots',
    order=RAIN_GROUP_ORDER,
    title='Shots nach Regen-Gruppe',
    xlabel='Regenbedingung',
    ylabel='Shots pro Team-Spiel',
    output_path=SHOTS_RAIN_VS_NO_RAIN_PATH,
    color=plot_colors['shots'],
)
shots_by_rain_group


## Vergleich: Chancenqualitaet vs. Tore

Die Korrelationsuebersicht vergleicht Wetter mit `xg_per_shot`, `shots`, `xg` und `team_score`. Damit wird sichtbar, ob Wetter eher mit Chancenqualitaet oder mit Toren beziehungsweise Abschlussvolumen zusammenhaengt.


In [ ]:
quality_vs_goals = correlation_summary.pivot(
    index='weather_variable',
    columns='metric',
    values='correlation',
).reset_index()
quality_vs_goals


## Regen-Summary

Die Tabelle zeigt Mittelwert und Median fuer Chancenqualitaet, xG, Shots und Tore getrennt nach Regenstatus.


In [ ]:
numeric_columns = rain_summary.select_dtypes(include='number').columns
rain_summary[numeric_columns] = rain_summary[numeric_columns].round(3)
rain_summary


## Output-Checks

Die Checks stellen sicher, dass alle geforderten BD-23-Artefakte geschrieben wurden und nicht leer sind.


In [ ]:
figure_paths = [
    TEMPERATURE_VS_XG_PER_SHOT_PATH,
    FEELS_LIKE_VS_XG_PER_SHOT_PATH,
    XG_BY_TEMPERATURE_GROUP_PATH,
    XG_RAIN_VS_NO_RAIN_PATH,
    SHOTS_RAIN_VS_NO_RAIN_PATH,
]

table_paths = [SUMMARY_PATH]

for path in figure_paths:
    assert path.exists(), f'Missing expected figure: {path}'
    assert path.stat().st_size > 1_000, f'Figure looks too small or empty: {path}'

for path in table_paths:
    assert path.exists(), f'Missing expected summary table: {path}'
    assert path.stat().st_size > 200, f'Summary table looks too small or empty: {path}'

pd.DataFrame({
    'artifact': [str(path.relative_to(ROOT)) for path in figure_paths + table_paths],
    'bytes': [path.stat().st_size for path in figure_paths + table_paths],
})


## Interpretation fuer BD-23

Die Auswertung trennt Chancenqualitaet von Abschlussvolumen: `xg_per_shot` beschreibt die durchschnittliche Qualitaet eines Schusses, `shots` die Anzahl der Abschluesse und `xg` die Kombination aus beidem. Die Regenvergleiche sind als grobe Gruppenvergleiche zu lesen, weil nur wenige Team-Spiele mehr als 0.5 mm Niederschlag haben.


In [ ]:
interpretation = {
    'team_matches': int(len(analysis_df)),
    'rain_threshold_mm': RAIN_THRESHOLD_MM,
    'low_or_no_rain_team_matches': int((analysis_df['rain_mm'] <= RAIN_THRESHOLD_MM).sum()),
    'measurable_rain_team_matches': int((analysis_df['rain_mm'] > RAIN_THRESHOLD_MM).sum()),
    'zero_xg_team_matches': int((analysis_df['xg'] == 0).sum()),
    'zero_shot_team_matches': int((analysis_df['shots'] == 0).sum()),
    'zero_temperature_team_matches': int((analysis_df['temperature_c'] == 0).sum()),
    'temperature_xg_per_shot_correlation': temperature_quality_relation['correlation'],
    'feels_like_xg_per_shot_correlation': feels_like_quality_relation['correlation'],
    'rain_mm_xg_correlation': float(correlation_summary.loc[
        (correlation_summary['weather_variable'] == 'rain_mm')
        & (correlation_summary['metric'] == 'team_xg'),
        'correlation',
    ].iloc[0]),
    'rain_mm_shots_correlation': float(correlation_summary.loc[
        (correlation_summary['weather_variable'] == 'rain_mm')
        & (correlation_summary['metric'] == 'shot_volume'),
        'correlation',
    ].iloc[0]),
    'strongest_weather_quality_correlation': round(float(correlation_summary.loc[
        correlation_summary['metric'].eq('chance_quality_xg_per_shot'), 'correlation'
    ].abs().max()), 3),
    'highest_xg_temperature_group': xg_by_temperature_group.sort_values('mean', ascending=False).iloc[0]['temperature_group'],
    'low_or_no_rain_xg_mean': float(xg_by_rain_group.loc[xg_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[0], 'mean'].iloc[0]),
    'measurable_rain_xg_mean': float(xg_by_rain_group.loc[xg_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[1], 'mean'].iloc[0]),
    'low_or_no_rain_shots_mean': float(shots_by_rain_group.loc[shots_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[0], 'mean'].iloc[0]),
    'measurable_rain_shots_mean': float(shots_by_rain_group.loc[shots_by_rain_group['rain_group'] == RAIN_GROUP_ORDER[1], 'mean'].iloc[0]),
}
interpretation


## BD-24 Regressionsmodell vorbereiten

Das vorbereitete Modell-Dataset folgt der Modellidee `team_xg ~ temperature + feels_like + rain + elo_diff + tournament`. Im Gold-Dataset heisst `team_xg` technisch `xg`; fuer das Modell wird deshalb eine klar benannte Kopie `team_xg` erzeugt. Als Tournament-Variable wird `short_name` verwendet, weil diese Spalte den Projekt-Scope kompakt und auswertbar beschreibt.


In [ ]:
model_identifier_columns = [
    'match_id',
    'team_id',
    'team_name',
    'opponent_team_name',
    'short_name',
]
model_target_columns = [
    'team_xg',
    'xg_per_shot',
    'pass_completion_rate',
    'pressures',
]
numeric_feature_columns = [
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'elo_diff',
]
categorical_feature_columns = [
    'tournament',
]

model_source_columns = model_identifier_columns + [
    'xg',
    'xg_per_shot',
    'pass_completion_rate',
    'pressures',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'elo_diff',
]
missing_model_source_columns = [column for column in model_source_columns if column not in team_match_df.columns]
assert not missing_model_source_columns, f'Missing model source columns: {missing_model_source_columns}'

model_base_df = team_match_df[model_source_columns].copy()
model_base_df['team_xg'] = model_base_df['xg']
model_base_df['tournament'] = model_base_df['short_name']
model_base_df = model_base_df.drop(columns=['xg'])

model_base_df.head()


## Fehlende Werte behandeln

Die Regressionsvorbereitung entfernt nur Zeilen, die in Ziel- oder Feature-Spalten fehlende Werte haben. Dadurch bleibt die Team-Match-Struktur unverfaelscht, und der Drop ist transparent dokumentiert.


In [ ]:
model_required_columns = model_target_columns + numeric_feature_columns + categorical_feature_columns
missing_values = (
    model_base_df[model_required_columns]
    .isna()
    .sum()
    .rename_axis('variable')
    .reset_index(name='missing_values')
)
missing_values['role'] = missing_values['variable'].map(
    {column: 'target' for column in model_target_columns}
    | {column: 'numeric_feature' for column in numeric_feature_columns}
    | {column: 'categorical_feature' for column in categorical_feature_columns}
)
missing_values.to_csv(MODEL_MISSING_VALUES_PATH, index=False)

model_clean_df = model_base_df.dropna(subset=model_required_columns).copy()
removed_rows = len(model_base_df) - len(model_clean_df)

assert len(model_clean_df) > 0, 'Model dataset is empty after missing-value handling.'
assert not model_clean_df[model_required_columns].isna().any().any(), 'Model dataset still contains missing target or feature values.'
assert not model_clean_df.duplicated(['match_id', 'team_id']).any(), 'Expected one row per match/team pair.'

{
    'input_team_matches': int(len(model_base_df)),
    'model_team_matches': int(len(model_clean_df)),
    'removed_rows_with_missing_values': int(removed_rows),
}


## Kategoriale Variablen kodieren

`tournament` wird mit Dummy-Spalten abgebildet; eine Kategorie wird bewusst weggelassen (`drop_first=True`), damit das lineare Regressionsmodell keine redundant kodierten Tournament-Spalten enthaelt. Regen bleibt im Modell als kontinuierliche Variable `rain_mm`, weil eine harte Regen-Kategorie fuer die Modellierung zu grob waere.


In [ ]:
model_feature_df = model_clean_df[model_identifier_columns + model_target_columns + numeric_feature_columns].copy()

tournament_dummies = pd.get_dummies(
    model_clean_df['tournament'],
    prefix='tournament',
    drop_first=True,
    dtype=int,
)
model_dataset_df = pd.concat([model_feature_df, tournament_dummies], axis=1)

encoded_feature_columns = numeric_feature_columns + list(tournament_dummies.columns)
model_dataset_columns = model_identifier_columns + model_target_columns + encoded_feature_columns
model_dataset_df = model_dataset_df[model_dataset_columns]

assert set(model_target_columns).isdisjoint(encoded_feature_columns), 'Targets and features must be separated.'
assert not model_dataset_df[model_target_columns + encoded_feature_columns].isna().any().any(), 'Encoded model dataset contains missing values.'

model_dataset_df.to_csv(MODEL_DATASET_PATH, index=False)
model_dataset_df.head()


## Modellvariablen dokumentieren

Die Variable-Liste trennt Identifikatoren, Zielvariablen und Features. `team_xg` ist das Hauptziel fuer BD-24; `xg_per_shot`, `pass_completion_rate` und `pressures` bleiben als optionale Zielvariablen fuer spaetere Modellvergleiche vorbereitet.


In [ ]:
variable_rows = []
for column in model_identifier_columns:
    variable_rows.append({
        'variable': column,
        'role': 'identifier',
        'source': column,
        'description': 'Keeps team-match rows traceable; not used as model feature.',
    })

for column in model_target_columns:
    variable_rows.append({
        'variable': column,
        'role': 'target',
        'source': 'xg' if column == 'team_xg' else column,
        'description': 'Main target for regression.' if column == 'team_xg' else 'Optional target for later robustness checks.',
    })

for column in numeric_feature_columns:
    variable_rows.append({
        'variable': column,
        'role': 'numeric_feature',
        'source': column,
        'description': 'Weather feature.' if column != 'elo_diff' else 'Team-strength feature: team Elo minus opponent Elo.',
    })

for column in tournament_dummies.columns:
    variable_rows.append({
        'variable': column,
        'role': 'encoded_feature',
        'source': 'tournament',
        'description': 'Tournament dummy from short_name; first tournament category is the baseline.',
    })

model_variables_df = pd.DataFrame(variable_rows)
model_variables_df.to_csv(MODEL_VARIABLES_PATH, index=False)
model_variables_df


## BD-24 Output-Checks und Kurzfazit

Die Checks pruefen, ob das Modell-Dataset, die Variablendokumentation und der Missing-Value-Bericht geschrieben wurden. Damit ist der naechste Schritt vorbereitet: In BD-25 koennen Wetter-Features und Teamstaerke in mehreren Regressionsvarianten verglichen werden.


In [ ]:
bd24_table_paths = [
    MODEL_DATASET_PATH,
    MODEL_VARIABLES_PATH,
    MODEL_MISSING_VALUES_PATH,
]

for path in bd24_table_paths:
    assert path.exists(), f'Missing BD-24 output table: {path}'
    assert path.stat().st_size > 100, f'BD-24 output table looks too small or empty: {path}'

bd24_summary = {
    'model_rows': int(len(model_dataset_df)),
    'targets': model_target_columns,
    'features': encoded_feature_columns,
    'removed_rows_with_missing_values': int(removed_rows),
    'train_test_split': 'optional for explanatory analysis; needed for predictive validation',
    'outputs': [str(path.relative_to(ROOT)) for path in bd24_table_paths],
}
bd24_summary


## BD-25 Regressionsmodelle vergleichen

Die Modelle schaetzen `team_xg` mit unterschiedlichen Eingabegruppen. Wichtig: `Wetter + Elo` bedeutet nicht, dass Wetterwerte und Elo-Werte addiert werden. Es bedeutet nur, dass beide Variablengruppen gleichzeitig in derselben linearen Regression stehen.

Beispielhaft ist das kombinierte Modell: `team_xg = Achsenabschnitt + Wetter-Koeffizienten * Wettervariablen + Elo-Koeffizient * elo_diff`.

Verglichen wird mit Adjusted R2. Diese Kennzahl bestraft zusaetzliche Features. Dadurch sieht man, ob ein groesseres Modell wirklich mehr erklaert oder nur durch mehr Variablen minimal besser an die vorhandenen Daten angepasst ist.


In [ ]:
def fit_ols_model(dataframe, target_column, feature_columns, model_name):
    model_input = dataframe[[target_column] + feature_columns].dropna().copy()
    y = model_input[target_column].to_numpy(dtype=float)
    x = model_input[feature_columns].to_numpy(dtype=float)
    x_with_intercept = np.column_stack([np.ones(len(model_input)), x])

    coefficients, _, _, _ = np.linalg.lstsq(x_with_intercept, y, rcond=None)
    predictions = x_with_intercept @ coefficients
    residuals = y - predictions

    total_sum_squares = float(((y - y.mean()) ** 2).sum())
    residual_sum_squares = float((residuals ** 2).sum())
    r2 = 1 - residual_sum_squares / total_sum_squares if total_sum_squares else 0.0
    n_observations = len(model_input)
    n_features = len(feature_columns)
    adjusted_r2 = 1 - (1 - r2) * (n_observations - 1) / (n_observations - n_features - 1)
    rmse = float(np.sqrt((residuals ** 2).mean()))

    return {
        'model_name': model_name,
        'target': target_column,
        'n_observations': int(n_observations),
        'n_features': int(n_features),
        'r2': float(r2),
        'adjusted_r2': float(adjusted_r2),
        'rmse': rmse,
        'features': feature_columns,
        'coefficients': dict(zip(['intercept'] + feature_columns, coefficients)),
    }


def standardize_columns(dataframe, columns):
    standardized = dataframe[columns].astype(float).copy()
    for column in columns:
        standard_deviation = standardized[column].std(ddof=0)
        if standard_deviation == 0:
            standardized[column] = 0.0
        else:
            standardized[column] = (standardized[column] - standardized[column].mean()) / standard_deviation
    return standardized


## Modellvarianten

Die vier Varianten sind bewusst additiv aufgebaut: Wetter allein, Elo allein, beide zusammen und zuletzt die Turnier-Dummies als Kontextkontrolle. Dadurch laesst sich sehen, ob ein hoeherer Modellwert wirklich aus Wetter/Elo kommt oder nur Turnierunterschiede aufnimmt.


In [ ]:
temperature_features = ['temperature_c', 'feels_like_c']
rain_features = ['rain_mm']
weather_features = temperature_features + rain_features
elo_features = ['elo_diff']
tournament_features = list(tournament_dummies.columns)

model_variants = {
    'weather_only': weather_features,
    'elo_only': elo_features,
    'weather_plus_elo': weather_features + elo_features,
    'weather_plus_elo_plus_tournament': weather_features + elo_features + tournament_features,
}

model_results = [
    fit_ols_model(model_dataset_df, 'team_xg', feature_columns, model_name)
    for model_name, feature_columns in model_variants.items()
]

model_comparison_df = pd.DataFrame([
    {
        'model_name': result['model_name'],
        'target': result['target'],
        'n_observations': result['n_observations'],
        'n_features': result['n_features'],
        'r2': round(result['r2'], 4),
        'adjusted_r2': round(result['adjusted_r2'], 4),
        'rmse': round(result['rmse'], 4),
        'feature_set': ', '.join(result['features']),
    }
    for result in model_results
])
model_comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)
model_comparison_df


## Koeffizienten vorbereiten

Der Koeffizientenplot nutzt standardisierte Features aus dem vollstaendigen Modell, zeigt aber nur die fachlich direkt interpretierbaren Variablen: Teamstaerke und Wetter. Turnier-Dummies bleiben im Modell als Kontrollvariablen enthalten, werden aber nicht im Hauptplot gezeigt, weil sie nur relativ zur nicht dargestellten Referenzkategorie sinnvoll interpretierbar sind. Das ersetzt keine Kausalanalyse, macht aber Richtung und relative Staerke der Modellzusammenhaenge lesbarer.


In [ ]:
full_model_features = model_variants['weather_plus_elo_plus_tournament']
standardized_model_df = model_dataset_df[['team_xg']].copy()
standardized_model_df[full_model_features] = standardize_columns(model_dataset_df, full_model_features)
standardized_full_model = fit_ols_model(
    standardized_model_df,
    target_column='team_xg',
    feature_columns=full_model_features,
    model_name='weather_plus_elo_plus_tournament_standardized',
)

coefficient_rows = []
for result in model_results:
    for feature, coefficient in result['coefficients'].items():
        coefficient_rows.append({
            'model_name': result['model_name'],
            'feature': feature,
            'coefficient': round(float(coefficient), 6),
            'coefficient_type': 'raw',
        })

for feature, coefficient in standardized_full_model['coefficients'].items():
    coefficient_rows.append({
        'model_name': standardized_full_model['model_name'],
        'feature': feature,
        'coefficient': round(float(coefficient), 6),
        'coefficient_type': 'standardized',
    })

model_coefficients_df = pd.DataFrame(coefficient_rows)
model_coefficients_df.to_csv(MODEL_COEFFICIENTS_TABLE_PATH, index=False)
model_coefficients_df.tail(len(full_model_features) + 1)


## Erklaerungsbeitrag visualisieren

Die Grafik beantwortet: Verbessern Wetterdaten ein Regressionsmodell fuer `team_xg`, und wie stark ist dieser Beitrag im Vergleich zur Elo-Differenz?

Gezeigt wird die Aenderung im Adjusted R2 gegen ein passendes Vergleichsmodell. `Wetterdaten allein` und `Elo-Differenz allein` werden gegen ein Modell ohne Features verglichen. `Wetterdaten zusaetzlich zu Elo` wird gegen das reine Elo-Modell verglichen. Dadurch sieht man direkt, ob Wetter nach Kontrolle fuer Teamstaerke noch etwas beitraegt.


In [ ]:
def build_incremental_model_comparison(comparison_df):
    lookup = comparison_df.set_index('model_name')
    comparisons = [
        {
            'comparison': 'Wetterdaten allein',
            'base_model': 'null_model',
            'candidate_model': 'weather_only',
            'note': 'Temperatur, gefuehlte Temperatur und Regenmenge gegen Intercept-only-Baseline.',
        },
        {
            'comparison': 'Elo-Differenz allein',
            'base_model': 'null_model',
            'candidate_model': 'elo_only',
            'note': 'Teamstaerke relativ zum Gegner gegen Intercept-only-Baseline.',
        },
        {
            'comparison': 'Wetterdaten zusaetzlich zu Elo',
            'base_model': 'elo_only',
            'candidate_model': 'weather_plus_elo',
            'note': 'Zusatznutzen von Wetterdaten, nachdem Teamstaerke kontrolliert ist.',
        },
    ]

    rows = []
    for item in comparisons:
        candidate = lookup.loc[item['candidate_model']]
        if item['base_model'] == 'null_model':
            base_r2 = 0.0
            base_adjusted_r2 = 0.0
            base_rmse = np.nan
            base_features = 0
        else:
            base = lookup.loc[item['base_model']]
            base_r2 = float(base['r2'])
            base_adjusted_r2 = float(base['adjusted_r2'])
            base_rmse = float(base['rmse'])
            base_features = int(base['n_features'])

        candidate_r2 = float(candidate['r2'])
        candidate_adjusted_r2 = float(candidate['adjusted_r2'])
        candidate_rmse = float(candidate['rmse'])
        candidate_features = int(candidate['n_features'])

        rows.append({
            'comparison': item['comparison'],
            'base_model': item['base_model'],
            'candidate_model': item['candidate_model'],
            'n_added_features': candidate_features - base_features,
            'r2': round(candidate_r2, 4),
            'adjusted_r2': round(candidate_adjusted_r2, 4),
            'delta_r2': round(candidate_r2 - base_r2, 4),
            'delta_adjusted_r2': round(candidate_adjusted_r2 - base_adjusted_r2, 4),
            'delta_rmse': np.nan if np.isnan(base_rmse) else round(candidate_rmse - base_rmse, 4),
            'note': item['note'],
        })

    return pd.DataFrame(rows)


def save_model_r2_comparison(incremental_df, output_path):
    order = [
        'Wetterdaten allein',
        'Elo-Differenz allein',
        'Wetterdaten zusaetzlich zu Elo',
    ]
    labels = [
        'Wetterdaten\nallein',
        'Elo-Differenz\nallein',
        'Wetterdaten\nzusaetzlich zu Elo',
    ]
    plot_df = incremental_df.set_index('comparison').loc[order].reset_index()
    values = plot_df['delta_adjusted_r2'].astype(float).to_numpy()
    colors = np.where(values > 0, '#2f7d58', np.where(values < 0, '#b24c4c', '#6f6f6f'))

    fig, ax = plt.subplots(figsize=(10.5, 4.8), dpi=160)
    y_positions = np.arange(len(plot_df))
    bars = ax.barh(y_positions, values, color=colors, height=0.54)

    lower = min(values.min(), 0.0)
    upper = max(values.max(), 0.0)
    padding = max((upper - lower) * 0.20, 0.003)
    ax.set_xlim(lower - padding, upper + padding)
    ax.axvline(0, color='#222222', linewidth=1)

    for bar, value in zip(bars, values):
        offset = 0.0011 if value >= 0 else -0.0011
        ha = 'left' if value >= 0 else 'right'
        ax.text(value + offset, bar.get_y() + bar.get_height() / 2, f'{value:+.4f}', va='center', ha=ha, fontsize=9)

    ax.set_title('Erklaerungsbeitrag fuer team_xG', loc='left', fontweight='bold', pad=30)
    ax.text(0, 1.03, 'Delta Adjusted R2: hoeher ist besser; links von 0 verschlechtert das Modell nach Feature-Strafe.',
            transform=ax.transAxes, ha='left', va='bottom', fontsize=10, color='#444444')
    ax.set_xlabel('Delta Adjusted R2 gegen Vergleichsmodell')
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()


model_incremental_comparison_df = build_incremental_model_comparison(model_comparison_df)
model_incremental_comparison_df.to_csv(MODEL_INCREMENTAL_COMPARISON_PATH, index=False)
save_model_r2_comparison(model_incremental_comparison_df, MODEL_R2_COMPARISON_PATH)
model_incremental_comparison_df


**Kurzfazit:** Die Elo-Differenz liefert den klaren Erklaerungsbeitrag fuer `team_xG` (`+0.0738`). Wetterdaten allein liegen leicht unter 0 (`-0.0033`) und verbessern das Elo-Modell nicht (`-0.0029`). Damit liefern die Wetterdaten in diesem Modell keinen robusten zusaetzlichen Erklaerungsnutzen.


## Koeffizientenplot

Positive Werte stehen fuer einen hoeheren erwarteten Team-xG-Wert, negative Werte fuer einen niedrigeren. Die Turnier-Dummies sind relativ zur ausgelassenen Basiskategorie zu lesen.


In [ ]:
def save_coefficient_plot(coefficients_df, output_path):
    shown_features = ['elo_diff', 'rain_mm', 'temperature_c', 'feels_like_c']
    feature_labels = {
        'elo_diff': 'Elo-Differenz',
        'rain_mm': 'Regenmenge (mm)',
        'temperature_c': 'Temperatur (C)',
        'feels_like_c': 'Gefuehlte Temperatur (C)',
    }
    plot_df = coefficients_df.loc[
        (coefficients_df['model_name'] == 'weather_plus_elo_plus_tournament_standardized')
        & (coefficients_df['feature'].isin(shown_features))
    ].copy()
    plot_df['feature_label'] = plot_df['feature'].map(feature_labels)
    plot_df = plot_df.sort_values('coefficient')
    colors = np.where(plot_df['coefficient'] >= 0, '#2f7d58', '#b24c4c')

    fig, ax = plt.subplots(figsize=(11, 5.4), dpi=160)
    bars = ax.barh(plot_df['feature_label'], plot_df['coefficient'], color=colors)
    ax.axvline(0, color='#222222', linewidth=1)
    ax.set_title('Einfluss interpretierbarer Modellvariablen auf team_xG', loc='left', fontweight='bold', pad=18)
    ax.set_xlabel('Standardisierter Koeffizient (positiv = hoehere vorhergesagte team_xG)')
    ax.set_xlim(min(-0.13, plot_df['coefficient'].min() - 0.04), plot_df['coefficient'].max() + 0.06)
    ax.grid(axis='x', alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)

    for bar, value in zip(bars, plot_df['coefficient']):
        label_x = value + 0.01 if value >= 0 else value - 0.01
        alignment = 'left' if value >= 0 else 'right'
        ax.text(label_x, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center', ha=alignment, fontsize=9)

    note = (
        'Lesart: Gruene Balken erhoehen die Modellvorhersage, rote Balken senken sie. '
        'Die Laenge zeigt die relative Staerke nach Standardisierung.\n'
        'Turnier-Dummies sind im Modell nur Kontrollvariablen und werden deshalb hier nicht gezeigt.'
    )
    fig.text(0.01, 0.01, note, ha='left', va='bottom', fontsize=9, color='#444444')
    fig.subplots_adjust(left=0.25, right=0.98, bottom=0.24, top=0.87)
    fig.savefig(output_path, bbox_inches='tight')
    plt.show()

    return plot_df[['model_name', 'feature', 'coefficient', 'coefficient_type']]


standardized_coefficients_for_plot = save_coefficient_plot(model_coefficients_df, MODEL_COEFFICIENTS_PATH)
standardized_coefficients_for_plot


**Kurzfazit:** Im Gesamtmodell ist die Elo-Differenz der deutlich staerkste interpretierbare Praediktor. Die Wettervariablen sind im Vergleich klein: Regenmenge liegt praktisch bei 0, Temperatur und gefuehlte Temperatur zeigen kleine negative Koeffizienten.


## BD-25 Output-Checks und Fazit

Die Checks pruefen die drei geforderten BD-25-Artefakte und die zusaetzliche Koeffiziententabelle. Das Fazit wird aus den berechneten Modellwerten abgeleitet, damit es beim erneuten Ausfuehren konsistent bleibt.


In [ ]:
bd25_output_paths = [
    MODEL_COMPARISON_PATH,
    MODEL_INCREMENTAL_COMPARISON_PATH,
    MODEL_COEFFICIENTS_TABLE_PATH,
    MODEL_R2_COMPARISON_PATH,
    MODEL_COEFFICIENTS_PATH,
]
for path in bd25_output_paths:
    assert path.exists(), f'Missing BD-25 output: {path}'
    assert path.stat().st_size > 100, f'BD-25 output looks too small or empty: {path}'

assert len(model_comparison_df) >= 4, 'BD-25 requires weather and Elo model variants.'
assert {'r2', 'adjusted_r2'}.issubset(model_comparison_df.columns), 'Model comparison must contain R2 metrics.'
assert 'delta_adjusted_r2' in model_incremental_comparison_df.columns, 'Incremental comparison must show delta Adjusted R2.'

best_model = model_comparison_df.sort_values('adjusted_r2', ascending=False).iloc[0]
weather_adjusted_r2 = float(model_comparison_df.loc[model_comparison_df['model_name'] == 'weather_only', 'adjusted_r2'].iloc[0])
elo_adjusted_r2 = float(model_comparison_df.loc[model_comparison_df['model_name'] == 'elo_only', 'adjusted_r2'].iloc[0])
weather_after_elo_delta_adjusted_r2 = float(model_incremental_comparison_df.loc[
    model_incremental_comparison_df['comparison'] == 'Wetterdaten zusaetzlich zu Elo',
    'delta_adjusted_r2',
].iloc[0])

bd25_summary = {
    'model_rows': int(len(model_dataset_df)),
    'model_variants': model_comparison_df['model_name'].tolist(),
    'best_model_by_adjusted_r2': best_model['model_name'],
    'best_adjusted_r2': float(best_model['adjusted_r2']),
    'weather_adjusted_r2': weather_adjusted_r2,
    'elo_adjusted_r2': elo_adjusted_r2,
    'weather_after_elo_delta_adjusted_r2': weather_after_elo_delta_adjusted_r2,
    'interpretation': (
        'Elo beschreibt deutlich mehr Varianz in team_xg als Wetterdaten. Wetterdaten liefern nach Kontrolle fuer Elo keinen robusten zusaetzlichen Erklaerungsbeitrag.'
    ),
    'outputs': [str(path.relative_to(ROOT)) for path in bd25_output_paths],
}
bd25_summary
